Визуализация для EDA и диагностики модели — практикум

**Идея:** каждое визуальное действие сопровождаем вопросом и выводом. Без короткой интерпретации шаг не засчитывается.

**Опорные материалы:** смотрите примеры из `pandas_matplotlib:seaborn.ipynb` и `eda-feat-engeen.ipynb`.

**Что делаем:**
- Быстрый обзор данных и признаков.
- Диагностика распределений, выбросов и корреляций.
- Проверка модели: кривые обучения/валидации, остатки, важность признаков.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    confusion_matrix, roc_curve, precision_recall_curve, roc_auc_score,
    mean_squared_error, r2_score
)

import warnings
warnings.filterwarnings('ignore')

# Настройка стиля
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


### Шпаргалка по практическим шагам
1. Визуальный скетч данных (pairplot/гистограммы).
2. Проверка баланса классов или диапазона целевой переменной.
3. Диагностика модели: ошибки на фолдах, остатки, важность признаков.
4. Мини-отчёт: 3–5 графиков + 3–5 выводов.


---
## 1. EDA-визуализации (краткий обзор)

Подробный EDA рассмотрен в `eda-feat-engeen.ipynb`. Здесь — быстрый обзор ключевых графиков.


In [ ]:
# Создаём синтетический датасет
np.random.seed(RANDOM_STATE)
n = 500

df = pd.DataFrame({
    'age': np.random.normal(35, 12, n).clip(18, 70),
    'income': np.random.exponential(50000, n),
    'credit_score': np.random.normal(650, 80, n).clip(300, 850),
    'years_employed': np.random.exponential(5, n).clip(0, 30),
    'target': np.random.randint(0, 2, n)
})

# Добавляем корреляцию с таргетом
df.loc[df['target'] == 1, 'income'] *= 1.5
df.loc[df['target'] == 1, 'credit_score'] += 50

print("Данные:")
print(df.head())


### 1.1 Распределения признаков


In [ ]:
# Гистограммы всех числовых признаков
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
numeric_cols = ['age', 'income', 'credit_score', 'years_employed']

for ax, col in zip(axes.flatten(), numeric_cols):
    # Гистограмма с KDE
    sns.histplot(data=df, x=col, hue='target', kde=True, ax=ax, palette='Set1')
    ax.set_title(f'Распределение: {col}')
    ax.legend(title='Target', labels=['0', '1'])

plt.tight_layout()
plt.show()


### 1.2 Box plots и Violin plots

Полезны для обнаружения выбросов и сравнения распределений между классами.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
sns.boxplot(data=df, x='target', y='income', ax=axes[0], palette='Set2')
axes[0].set_title('Box plot: Income по классам')
axes[0].set_xlabel('Target')
axes[0].set_ylabel('Income')

# Violin plot — показывает форму распределения
sns.violinplot(data=df, x='target', y='credit_score', ax=axes[1], palette='Set2')
axes[1].set_title('Violin plot: Credit Score по классам')
axes[1].set_xlabel('Target')
axes[1].set_ylabel('Credit Score')

plt.tight_layout()
plt.show()


---
## 2. Корреляционная матрица

Показывает линейные зависимости между признаками.


### Практика: сделайте диагностику распределений
- Постройте минимум два графика: гистограммы и box/violin для ключевых признаков.
- Подпишите оси и добавьте короткий вывод под каждым графиком.


In [ ]:
# TODO: ваши графики распределений
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# sns.histplot(data=df, x="...", hue="target", kde=True, ax=axes[0])
# sns.boxplot(data=df, x="target", y="...", ax=axes[1])
# plt.show()
# print("Вывод: ...")


In [ ]:
# Корреляционная матрица
corr_matrix = df.corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Маска для верхнего треугольника
sns.heatmap(
    corr_matrix, 
    mask=mask,
    annot=True, 
    fmt='.2f', 
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    square=True
)
plt.title('Корреляционная матрица')
plt.tight_layout()
plt.show()

print("Корреляция с таргетом:")
print(corr_matrix['target'].sort_values(ascending=False))


---
## 3. Диагностика классификации

После обучения модели важно понять, где она ошибается.


In [ ]:
# Обучаем модель классификации
X = df[['age', 'income', 'credit_score', 'years_employed']]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")


### 3.1 Confusion Matrix


In [ ]:
def plot_confusion_matrix(y_true, y_pred, normalize=False, title='Confusion Matrix'):
    """Функция для отрисовки confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        fmt = '.2%'
    else:
        fmt = 'd'
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt=fmt, cmap='Blues',
        xticklabels=['Pred 0', 'Pred 1'],
        yticklabels=['True 0', 'True 1']
    )
    plt.title(title)
    plt.ylabel('Истинный класс')
    plt.xlabel('Предсказанный класс')
    plt.tight_layout()
    plt.show()

# Обычная confusion matrix
plot_confusion_matrix(y_test, y_pred, normalize=False, title='Confusion Matrix (абсолютные значения)')

# Нормализованная
plot_confusion_matrix(y_test, y_pred, normalize=True, title='Confusion Matrix (нормализованная)')


### 3.2 ROC-кривая

ROC показывает trade-off между True Positive Rate и False Positive Rate при разных порогах.


### Практика: проверяем модель глазами
- Обучите простую модель (логистическая регрессия/дерево).
- Постройте кривые обучения или калибровочные/ROC-кривые.
- Посмотрите на важность признаков или shapley values (по желанию).


In [ ]:
# TODO: визуальная диагностика модели
# from sklearn.model_selection import learning_curve
# train_sizes, train_scores, val_scores = learning_curve(model, X, y, cv=5)
# plt.plot(train_sizes, train_scores.mean(axis=1), label="train")
# plt.plot(train_sizes, val_scores.mean(axis=1), label="val")
# plt.legend(); plt.xlabel("train size"); plt.ylabel("score")
# plt.show()
# print("Вывод: ...")


In [ ]:
def plot_roc_curve(y_true, y_pred_proba, title='ROC Curve'):
    """Функция для отрисовки ROC-кривой."""
    fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
    auc = roc_auc_score(y_true, y_pred_proba)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {auc:.3f})')
    plt.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random (AUC = 0.5)')
    plt.fill_between(fpr, tpr, alpha=0.3)
    
    plt.xlabel('False Positive Rate (1 - Specificity)')
    plt.ylabel('True Positive Rate (Sensitivity)')
    plt.title(title)
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_roc_curve(y_test, y_pred_proba)


### 3.3 Precision-Recall кривая

Лучше подходит для несбалансированных данных, чем ROC.


In [ ]:
def plot_pr_curve(y_true, y_pred_proba, title='Precision-Recall Curve'):
    """Функция для отрисовки PR-кривой."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_pred_proba)
    
    # Базовая линия — доля положительного класса
    baseline = y_true.mean()
    
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, 'b-', linewidth=2, label='PR Curve')
    plt.axhline(y=baseline, color='r', linestyle='--', linewidth=1, 
                label=f'Baseline (Precision = {baseline:.3f})')
    plt.fill_between(recall, precision, alpha=0.3)
    
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(title)
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.xlim([0, 1])
    plt.ylim([0, 1])
    plt.tight_layout()
    plt.show()

plot_pr_curve(y_test, y_pred_proba)


---
## 4. Диагностика регрессии


In [ ]:
# Создаём данные для регрессии
X_reg, y_reg = make_regression(
    n_samples=500, n_features=10, n_informative=5,
    noise=30, random_state=RANDOM_STATE
)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=RANDOM_STATE
)

reg = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
reg.fit(X_train_reg, y_train_reg)
y_pred_reg = reg.predict(X_test_reg)

print(f"R²: {r2_score(y_test_reg, y_pred_reg):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_reg, y_pred_reg)):.4f}")


### 4.1 Predicted vs Actual

В идеале точки должны лежать на диагонали y = x.


### Итог: визуальный отчёт
Соберите 4–6 графиков: распределения, корреляции, диагностика модели.
После каждого графика добавьте одну фразу-инсайт.
В конце сделайте общий список наблюдений: что важно для модели, где слабые места данных.


In [ ]:
def plot_predicted_vs_actual(y_true, y_pred, title='Predicted vs Actual'):
    """График предсказаний против истинных значений."""
    plt.figure(figsize=(8, 8))
    
    # Scatter plot
    plt.scatter(y_true, y_pred, alpha=0.5, edgecolors='none')
    
    # Идеальная линия
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Идеальная модель')
    
    plt.xlabel('Истинные значения')
    plt.ylabel('Предсказанные значения')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.axis('equal')
    plt.tight_layout()
    plt.show()

plot_predicted_vs_actual(y_test_reg, y_pred_reg)


### 4.2 Residual Plots

Остатки (residuals) = y_true - y_pred. В идеале должны быть случайно разбросаны вокруг 0.


In [ ]:
def plot_residuals(y_true, y_pred, title='Residual Plot'):
    """График остатков."""
    residuals = y_true - y_pred
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Residuals vs Predicted
    axes[0].scatter(y_pred, residuals, alpha=0.5, edgecolors='none')
    axes[0].axhline(y=0, color='r', linestyle='--', linewidth=2)
    axes[0].set_xlabel('Предсказанные значения')
    axes[0].set_ylabel('Остатки (y_true - y_pred)')
    axes[0].set_title('Остатки vs Предсказания')
    axes[0].grid(True, alpha=0.3)
    
    # Гистограмма остатков
    axes[1].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
    axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2)
    axes[1].set_xlabel('Остатки')
    axes[1].set_ylabel('Частота')
    axes[1].set_title('Распределение остатков')
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()
    
    print(f"Средние остатки: {residuals.mean():.4f} (должны быть ~0)")
    print(f"Std остатков: {residuals.std():.4f}")

plot_residuals(y_test_reg, y_pred_reg)


---
## 5. Feature Importance

Понимание того, какие признаки важны для модели.


In [ ]:
def plot_feature_importance(model, feature_names, top_n=10, title='Feature Importance'):
    """График важности признаков для древесных моделей."""
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1][:top_n]
    
    plt.figure(figsize=(10, 6))
    plt.barh(range(len(indices)), importances[indices][::-1], align='center')
    plt.yticks(range(len(indices)), [feature_names[i] for i in indices[::-1]])
    plt.xlabel('Важность')
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Для классификации
feature_names_clf = ['age', 'income', 'credit_score', 'years_employed']
plot_feature_importance(clf, feature_names_clf, title='Feature Importance (классификация)')


In [ ]:
# Для регрессии
feature_names_reg = [f'feature_{i}' for i in range(10)]
plot_feature_importance(reg, feature_names_reg, title='Feature Importance (регрессия)')


---
## 6. Практика

Соберите все графики в одну функцию для полной диагностики.


In [ ]:
# TODO: Создайте функцию full_classification_diagnostics, которая принимает:
# - y_true, y_pred, y_pred_proba
# И выводит все графики: confusion matrix, ROC, PR curve

def full_classification_diagnostics(y_true, y_pred, y_pred_proba):
    """Полная диагностика классификации."""
    # ВАШ КОД ЗДЕСЬ
    pass

# Проверка
# full_classification_diagnostics(y_test, y_pred, y_pred_proba)


In [ ]:
# SOLUTION
def full_classification_diagnostics(y_true, y_pred, y_pred_proba):
    """Полная диагностика классификации."""
    fig = plt.figure(figsize=(15, 10))
    
    # Confusion Matrix
    ax1 = fig.add_subplot(2, 2, 1)
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
                xticklabels=['Pred 0', 'Pred 1'],
                yticklabels=['True 0', 'True 1'])
    ax1.set_title('Confusion Matrix')
    ax1.set_ylabel('Истинный класс')
    ax1.set_xlabel('Предсказанный класс')
    
    # ROC Curve
    ax2 = fig.add_subplot(2, 2, 2)
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    auc = roc_auc_score(y_true, y_pred_proba)
    ax2.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {auc:.3f})')
    ax2.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random')
    ax2.fill_between(fpr, tpr, alpha=0.3)
    ax2.set_xlabel('FPR')
    ax2.set_ylabel('TPR')
    ax2.set_title('ROC Curve')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # PR Curve
    ax3 = fig.add_subplot(2, 2, 3)
    precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)
    ax3.plot(recall, precision, 'b-', linewidth=2)
    ax3.axhline(y=y_true.mean(), color='r', linestyle='--', linewidth=1)
    ax3.fill_between(recall, precision, alpha=0.3)
    ax3.set_xlabel('Recall')
    ax3.set_ylabel('Precision')
    ax3.set_title('Precision-Recall Curve')
    ax3.grid(True, alpha=0.3)
    
    # Распределение вероятностей
    ax4 = fig.add_subplot(2, 2, 4)
    for label in [0, 1]:
        mask = y_true == label
        ax4.hist(y_pred_proba[mask], bins=20, alpha=0.6, label=f'Class {label}')
    ax4.set_xlabel('Predicted Probability')
    ax4.set_ylabel('Count')
    ax4.set_title('Распределение предсказанных вероятностей')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Проверка
full_classification_diagnostics(y_test, y_pred, y_pred_proba)


---
## Что дальше?

Теперь вы умеете:
- Строить EDA-визуализации для понимания данных
- Диагностировать модели классификации (CM, ROC, PR)
- Диагностировать модели регрессии (residuals, predicted vs actual)
- Визуализировать важность признаков

**Следующий шаг:** `04_end_to_end_tabular.ipynb` — полный пример решения табличной задачи от начала до конца.

**Также рекомендуем:**
- `eda-feat-engeen.ipynb` — подробный EDA и feature engineering
- `boostings_101.ipynb` — подбор гиперпараметров бустингов
